# Example: Static inverse free-boundary equilibrium calculations (in KSTAR)

---

Here we generate a KSTAR-like equilibrium using the machine geometry in the [FUSE.jl repository](https://github.com/ProjectTorreyPines/FUSE.jl).

The equilibrium/profile parameters are a mixture of number from the source files and **completely made up**; please experiment on your own and change them to more realistic values as you please!

Note that there are no passive conducting structures in this description!

### Import packages

In [ ]:
import matplotlib.pyplot as plt

import numpy as np

### Create the machine object

In [ ]:
# build machine

from freegsnke import build_machine

tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/KSTAR/KSTAR_active_coils.pickle",
    limiter_path="../machine_configs/KSTAR/KSTAR_limiter.pickle",
    wall_path="../machine_configs/KSTAR/KSTAR_wall.pickle",
)

In [ ]:
# plot the machine

fig, ax = plt.subplots(figsize=(8, 8), dpi=80)

tokamak.plot(axis=ax, show=False)  # plots the active coils
ax.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)  # plots the limiter

ax.grid(alpha=0.5)
ax.set_aspect("equal")
ax.set_xlim(0.4, 3.9)
ax.set_ylim(-2.6, 2.6)
ax.set_xlabel(r"Major radius, $R$ [m]")
ax.set_ylabel(r"Height, $Z$ [m]")

### Instantiate an equilibrium

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=1.2, Rmax=2.7,  # radial range
    Zmin=-1.5, Zmax=1.5,  # vertical range
    nx=129,               # number of radial grid points (needs to be of the form 2**n + 1)
    ny=129,               # number of vertical grid points (needs to be of the form 2**n + 1)
)

### Instantiate a profile object

In [ ]:
# initialise the profiles

from freegsnke.jtor_update import ConstrainPaxisIp

profiles = ConstrainPaxisIp(
    eq=eq,         # equilibrium object
    paxis=5.0e4,   # illustrative pressure on axis [Pa]
    Ip=0.8e6,      # plasma current from the FUSE KSTAR case [A]
    fvac=3.24,     # abs(B0) * R0 from the FUSE KSTAR case [T m]
    alpha_m=1.8,   # profile function parameter
    alpha_n=1.2,   # profile function parameter
)

### Load the static nonlinear solver

In [ ]:
from freegsnke import GSstaticsolver

GSStaticSolver = GSstaticsolver.NKGSsolver(eq, seed=42)

### Constraints

In [ ]:
# set the desired lower X-point and magnetic-axis locations
Rx, Zx = 1.40, -1.05
null_points = [[Rx,Rx], [Zx, -Zx]]

# constrain boundary points to the same flux surface
isoflux_set = np.array([[
    [1.31, 2.23, Rx, Rx],
    [0.0, 0.0, Zx, -Zx],
]])

from freegsnke.inverse import Inverse_optimizer

constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
)

### The inverse solve

In [ ]:
GSStaticSolver.inverse_solve(
    eq=eq,
    profiles=profiles,
    constrain=constrain,
    target_relative_tolerance=1e-5,
    target_relative_psit_update=1e-3,
    max_solving_iterations=100,
    l2_reg=np.array([1e-14] * 14),
    Picard_handover=1e-4,
    force_up_down_symmetric=True,
    verbose=True,    
)

In [ ]:
# refine with a forward solve (now the currents are known)
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=None, 
                     target_relative_tolerance=1e-9,
                     verbose=False, # suppress output
                     )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), dpi=80)

ax.grid(zorder=0, alpha=0.75)
ax.set_aspect("equal")
eq.tokamak.plot(axis=ax, show=False)  # plots the active coils
eq.plot(axis=ax, show=False)          # plots the equilibrium
constrain.plot(axis=ax, show=False)   # plots the constraints
ax.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)  # plots the limiter
# ax.set_xlim(1.15, 2.4)
# ax.set_ylim(-1.55, 1.55)
ax.set_xlabel(r"Major radius, $R$ [m]")
ax.set_ylabel(r"Height, $Z$ [m]")

In [ ]:
eq.tokamak.getCurrents()